In [ ]:
# '취업'에 대해 학습 시키도록 한다
# 네이버 지식인에서 취업과 관련된 질문들을 가져와서 학습시키기 전 단계까지 처리하려고 한다

# 네이버 지식인에서 취업과 관련된 질문 200개를 추출해서 가져오는 작업

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

def get_naver_question_list(keyword: str, max_pages: int = 20):
    url = 'https://kin.naver.com/search/list.naver'
    
    titles = []
    contents = []
    hrefs = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer': 'https://kin.naver.com/'
    }

    session = requests.Session()
    session.headers.update(headers)

    print(f"총 {max_pages * 10}개의 데이터 수집을 시작합니다... ")

    for page in range(1, max_pages + 1):
        page_num = (page - 1) * 10 + 1 
        
        params = {
            'where': 'qna',
            'query': keyword,
            'page': page_num
        }

        try:
            response = session.get(url, params=params)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'lxml')
            links_divs = soup.select('ul.basic1 > li')

            if not links_divs:
                print(f"{page}페이지에 더 이상 결과가 없습니다.")
                break

            for link_div in links_divs:
                title_link = link_div.select_one('dt > a')
                
                if title_link:
                    title = title_link.text.strip()
                    detail_url = title_link.get("href")

                    try:
                        detail_response = session.get(detail_url)
                        detail_response.raise_for_status()
                        detail_soup = BeautifulSoup(detail_response.text, 'lxml')

                        content_tag = detail_soup.select_one('.c-heading__content, .questionDetail')

                        if content_tag:
                            content = content_tag.text.strip() 
                        else:
                            content = "본문 없음"

                    except Exception as e:
                        content = "본문 수집 실패"

                    titles.append(title)
                    contents.append(content)
                    hrefs.append(detail_url)

                    time.sleep(0.1)
                    
            print(f"{page}페이지 수집 완료! (현재 {len(titles)}/200개)")

        except Exception as e:
            print(f"예외 발생 ({page}페이지) : {e}")
            break

    return pd.DataFrame({
        'title': titles,
        'content': contents,
        'href': hrefs
    })

def save_nl(df: pd.DataFrame, keyword: str):
    if df is None or df.empty:
        print("가져온 데이터가 없어서 저장을 취소합니다.")
        return

    df = df.drop_duplicates(subset=['href'], keep='first')
    
    mean_len = df['content'].str.len().mean()
    print(f"데이터 본문의 평균 길이는 [{mean_len}] 입니다.")
    
    df = df.loc[df['content'].str.len() > mean_len]
    print(f"본문의 길이가 [{mean_len}] 이상인 데이터들만 저장중입니다.")
    
    file_name = f'naver_kin_{keyword}_200질문.csv'
    df.to_csv(
        file_name,
        encoding="utf-8-sig",
        index=False
    )
    print(f"[{file_name}] 파일로 성공적으로 저장되었습니다!")


if __name__ == "__main__":
    keyword = input("키워드를 입력하세요 : ") 
    print(f"\n입력하신 키워드는 : {keyword} 입니다.\n")
    
    nl_df = get_naver_question_list(keyword, max_pages=20)
    save_nl(nl_df, keyword)

In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

def get_naver_question_list(keyword: str, target_count: int = 200, min_len: int = 50):
    url = 'https://kin.naver.com/search/list.naver'
    
    titles = []
    contents = []
    hrefs = []
    
    seen_hrefs = set()
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer': 'https://kin.naver.com/'
    }

    session = requests.Session()
    session.headers.update(headers)

    print(f"데이터 {target_count}개 수집을 시작합니다... ")

    page = 1
    
    no_more_data = False
    
    while len(titles) < target_count:
        
        if no_more_data:
            break
        
        params = {
            'where': 'qna',
            'query': keyword,
            'page': page
        }

        try:
            response = session.get(url, params=params)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'lxml')
            links_divs = soup.select('ul.basic1 > li')
            
            if not links_divs:
                print(f"{page}페이지에 더 이상 결과가 없습니다. 수집을 종료합니다.")
                no_more_data = True
                break
            
            page_results = 0
            
            for link_div in links_divs:
                if len(titles) >= target_count:
                    break
                    
                title_link = link_div.select_one('dt > a')
                
                if title_link:
                    detail_url = title_link.get("href")
                    
                    seen_hrefs.add(detail_url)
                    page_results += 1

                    try:
                        detail_response = session.get(detail_url)
                        detail_response.raise_for_status()
                        detail_soup = BeautifulSoup(detail_response.text, 'lxml')
                        
                        title_tag = detail_soup.select_one('.endTitleSection')
                        content_tag = detail_soup.select_one('.questionDetail')
                        
                        if title_tag:
                            blind_tag = title_tag.select_one('.blind')
                            if blind_tag:
                                blind_tag.extract()
                            title = title_tag.text.strip()
                        else:
                            title = "제목 없음"
                            
                        if content_tag:
                            content = content_tag.text.strip() 
                        else:
                            content = "본문 없음"

                    except Exception as e:
                        content = "본문 수집 실패"
                        
                    if len(content) < min_len:
                        continue

                    titles.append(title)
                    contents.append(content)
                    hrefs.append(detail_url)
                    seen_hrefs.add(detail_url)

                    time.sleep(0.1)
                    
            if page_results == 0:
                print(f"수집할 데이터가 없어 종료합니다. (마지막 페이지 도달)")
                no_more_data = True
                break
                   
            print(f"{page}페이지 수집 완료, 데이터: {len(titles)}/{target_count}개)")
            page += 1

        except Exception as e:
            print(f"예외 발생 ({page}페이지) : {e}")
            break

    return pd.DataFrame({
        'title': titles,
        'content': contents,
        'href': hrefs
    })

def save_nl(df: pd.DataFrame, keyword: str):
    if df is None or df.empty:
        print("가져온 데이터가 없어서 저장을 취소합니다.")
        return
    
    original_count = len(df)
    
    mean_len = df['content'].str.len().mean()
    print(f"데이터 본문의 평균 길이는 [{mean_len}]글자 입니다.")
    
    filtered_df = df.loc[df['content'].str.len() > mean_len]
    print(f"본문의 길이가 [{mean_len}]글자 이상인 데이터들만 저장중입니다.")
    
    final_count = len(filtered_df)
    deleted_count = original_count - final_count
    print(f"[{deleted_count}]개의 데이터들은 제거되어 [{final_count}]개의 데이터들이 남았습니다.")
    print(f"남은 데이터들을 저장하겠습니다.")
    
    file_name = f'kin_{keyword}_limit200.csv'
    df.to_csv(
        file_name,
        encoding="utf-8-sig",
        index=False
    )
    
    print(f"[{file_name}] 데이터가 성공적으로 저장되었습니다.")


if __name__ == "__main__":
    keyword = input("키워드를 입력하세요 : ") 
    print(f"입력하신 키워드는 : {keyword} 입니다.")
    
    nl_df = get_naver_question_list(keyword, target_count=200)
    save_nl(nl_df, keyword)

입력하신 키워드는 : 고양이 입니다.
데이터 200개 수집을 시작합니다... 
1페이지 수집 완료, 데이터: 8/200개)
2페이지 수집 완료, 데이터: 18/200개)
3페이지 수집 완료, 데이터: 27/200개)
4페이지 수집 완료, 데이터: 35/200개)
5페이지 수집 완료, 데이터: 44/200개)
6페이지 수집 완료, 데이터: 53/200개)
7페이지 수집 완료, 데이터: 62/200개)
8페이지 수집 완료, 데이터: 71/200개)
9페이지 수집 완료, 데이터: 79/200개)
10페이지 수집 완료, 데이터: 89/200개)
11페이지 수집 완료, 데이터: 98/200개)
12페이지 수집 완료, 데이터: 107/200개)
13페이지 수집 완료, 데이터: 116/200개)
14페이지 수집 완료, 데이터: 125/200개)
15페이지 수집 완료, 데이터: 134/200개)
16페이지 수집 완료, 데이터: 143/200개)
17페이지 수집 완료, 데이터: 152/200개)
18페이지 수집 완료, 데이터: 162/200개)
19페이지 수집 완료, 데이터: 171/200개)
20페이지 수집 완료, 데이터: 180/200개)
21페이지 수집 완료, 데이터: 186/200개)
22페이지 수집 완료, 데이터: 193/200개)
23페이지 수집 완료, 데이터: 200/200개)
데이터 본문의 평균 길이는 [200.72]글자 입니다.
본문의 길이가 [200.72]글자 이상인 데이터들만 저장중입니다.
[131]개의 데이터들은 제거되어 [69]개의 데이터들이 남았습니다.
남은 데이터들을 저장하겠습니다.
[kin_고양이_limit200.csv] 데이터가 성공적으로 저장되었습니다.
